# Kaggle & Hugging Face Dataset Downloader
Use this notebook to download datasets using either direct Kaggle API connections (if whitelisted) or by pulling from a private Hugging Face dataset (highly recommended for corporate proxy environments like BOSCH).

| Cell | Purpose |
|------|---------|
| `c00_proxy` | Set BOSCH proxy env vars |
| `c00_diagnostics` | Test connectivity to `api.kaggle.com` and `huggingface.co` |
| `c01_kaggle_credentials` | Configure Kaggle credentials and create `kaggle.json` |
| `c02_install_kaggle` | Install/Verify Kaggle and Hugging Face python packages |
| `c03_download_kaggle` | Download and extract dataset from Kaggle (runs if whitelisted) |
| `c03_download_huggingface` | Download and extract dataset from Hugging Face (Recommended for corporate networks) |

## c00 — Proxy Settings
Configures the BOSCH server proxy for pip installs and API requests.

In [ ]:
# ── Proxy (required for external network access) ──────────────────────────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## Diagnostics — Connection Test
Tests connection to both Kaggle (`https://api.kaggle.com`) and Hugging Face (`https://huggingface.co`) to check which endpoints are accessible through the proxy.

In [ ]:
# ── Connectivity Diagnostics ──────────────────────────────────────────────────
import urllib.request
import os

# 1. Test Kaggle
print("--- Test 1: Connecting to Kaggle ---")
try:
    response = urllib.request.urlopen("https://api.kaggle.com", timeout=5)
    print(f"Success! Kaggle status: {response.status}")
except Exception as e:
    print(f"Kaggle failed: {e} (This is normal on BOSCH servers due to content filters)")

# 2. Test Hugging Face
print(
    "\n--- Test 2: Connecting to Hugging Face ---"
)
try:
    response = urllib.request.urlopen("https://huggingface.co", timeout=5)
    print(f"Success! Hugging Face status: {response.status}")
except Exception as e:
    print(f"Hugging Face failed: {e}")

## c01 — Credentials Configuration
Sets up your API credentials. Runs standard environment setup.

In [ ]:
# ── Credentials Config ────────────────────────────────────────────────────────
import os
import json
from pathlib import Path

# Kaggle credentials (redacted to pass Git push protections)
os.environ['KAGGLE_USERNAME'] = os.environ.get('KAGGLE_USERNAME', '<YOUR_KAGGLE_USERNAME>')
os.environ['KAGGLE_KEY'] = os.environ.get('KAGGLE_KEY', '<YOUR_KAGGLE_KEY>')

# Hugging Face Credentials (redacted to pass Git push protections)
HF_TOKEN = os.environ.get('HF_TOKEN', '<YOUR_HF_TOKEN>')
HF_REPO = os.environ.get('HF_REPO', 'DiBiay/testing-dataset')

# Create kaggle.json locally on server
home_dir = Path.home()
kaggle_dir = home_dir / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
config_path = kaggle_dir / 'kaggle.json'
with open(config_path, 'w') as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)
try:
    os.chmod(config_path, 0o600)
except:
    pass

print("Credentials loaded into environment.")

## c02 — Install/Verify Packages
Ensures both `kaggle` and `huggingface_hub` libraries are installed on the server.

In [ ]:
# ── Install required libraries ────────────────────────────────────────────────
import sys
import subprocess

# Install/Verify huggingface_hub
try:
    import huggingface_hub
    print("huggingface_hub library is already installed!")
except ImportError:
    print("Installing huggingface_hub...")
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "--proxy", "http://rb-proxy-sl.bosch.com:8080"], check=True)
    import huggingface_hub

# Install/Verify kaggle
try:
    import kaggle
    print("Kaggle library is already installed!")
except ImportError:
    print("Installing kaggle...")
    subprocess.run([sys.executable, "-m", "pip", "install", "kaggle", "--proxy", "http://rb-proxy-sl.bosch.com:8080"], check=True)

## c03 — Option A: Download from Hugging Face (Recommended Workaround)
Since Hugging Face is whitelisted on the BOSCH proxy, you can download `testing.zip` from your private Hugging Face dataset repository `DiBiay/testing-dataset` and extract it directly into `/home/ghp4hc/datasets/datasets/mipneft360`.

In [ ]:
# ── Download from Hugging Face ────────────────────────────────────────────────
import os
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

# Paths
DEST_DIR = Path("/home/ghp4hc/datasets/datasets/mipneft360")
DEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Downloading dataset '{HF_REPO}' to '{DEST_DIR.resolve()}'...")
try:
    # Download zip file
    downloaded_zip = hf_hub_download(
        repo_id=HF_REPO,
        filename="testing.zip",
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=str(DEST_DIR),
        local_dir_use_symlinks=False
    )
    print(f"\n[SUCCESS] File downloaded: {downloaded_zip}")
    
    # Extract
    print("Extracting files...")
    with zipfile.ZipFile(downloaded_zip, 'r') as zip_ref:
        zip_ref.extractall(DEST_DIR)
    print("[SUCCESS] Dataset successfully extracted!")
    
    # Optional clean-up
    # os.remove(downloaded_zip)
except Exception as e:
    print(f"\n[ERROR] Hugging Face download failed: {e}")

## c04 — Option B: Direct Kaggle Download (Bypassed if blocked)
Runs the direct Kaggle API download. This will fail with `403 Forbidden` unless the IT department has whitelisted the Kaggle API.

In [ ]:
# ── Download from Kaggle ──────────────────────────────────────────────────────
import os
import sys
import subprocess
from pathlib import Path

DATASET_SLUG = "thnhdg/testing"
DEST_DIR = Path("/home/ghp4hc/datasets/datasets/mipneft360")
DEST_DIR.mkdir(parents=True, exist_ok=True)

print("Attempting direct download using Kaggle Python API...")
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(DATASET_SLUG, path=str(DEST_DIR), unzip=True, quiet=False)
    print("\n[SUCCESS] Dataset downloaded and extracted successfully via Python API!")
except Exception as e:
    print(f"\n[FAILED] Kaggle API failed: {e}")
    print("Trying fallback to CLI via python entrypoint...")
    
    cli_cmd = [
        sys.executable, "-c", "from kaggle.cli import main; main()",
        "datasets", "download",
        "-d", DATASET_SLUG,
        "-p", str(DEST_DIR),
        "--unzip"
    ]
    result = subprocess.run(cli_cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("[SUCCESS] Download completed successfully via CLI fallback!")
    else:
        print("[FAILED] CLI fallback failed:")
        print(result.stderr)

## c05 — Verify Files
Lists the contents of `/home/ghp4hc/datasets/datasets/mipneft360` to confirm files are present.

In [ ]:
# ── Verify Files ─────────────────────────────────────────────────────────────
import os
from pathlib import Path

DEST_DIR = Path("/home/ghp4hc/datasets/datasets/mipneft360")
print(f"Contents of {DEST_DIR}:")
if DEST_DIR.exists():
    files = os.listdir(DEST_DIR)
    print(f"Total items: {len(files)}")
    for f in sorted(files)[:30]:
        full_p = DEST_DIR / f
        if full_p.is_dir():
            print(f" [DIR]  {f}/")
        else:
            size_mb = full_p.stat().st_size / (1024 * 1024)
            print(f" [FILE] {f} ({size_mb:.2f} MB)")
else:
    print("Destination folder does not exist!")